# 01 — Data cleaning

Turns the three raw files listed in data/README.md into the two
datasets used by `02_benchmark.ipynb`.
| Input (`data/raw/`) | Output (`data/clean/`) |
|---|---|
| `Moffitt_and_Bambah-Mukku_et_al_merfish_all_cells.csv` | `cells_cleaned.csv` — 874,768 cells × 164 columns |
| `cellboundaries_example_animal.csv` | `cells_boundaries_clean.csv` — 35,522 cells with repaired polygons |
| `merfish_barcodes_example.csv` | `barcodes_cleaned.csv` — not used for modelling |

Steps:

- **Step 0** — Paths and the shared truncation test
- **Step 1** — Clean the cell, boundary and barcode tables
- **Step 2** — Link cells to boundaries (64,312 cells)
- **Step 3** — Diagnose truncated boundary coordinates
- **Step 4** — Distribution of X/Y count differences after the repair
- **Step 5** — Repair boundaries, keep differences ≤ 1, save (35,522 cells)

Steps 3 and 4 only print diagnostics; they produce the counts reported in the
paper's data-cleaning section and change no data.

## Step 0 — Paths

In [ ]:
# =============================================================
# STEP 0 — Paths and shared helpers
# Run from the repository root or from notebooks/. Raw files go in
# data/raw/ (see data/README.md); cleaned files are written to data/clean/.
# =============================================================
from pathlib import Path
import pandas as pd
import numpy as np

ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW   = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

PATH_CELLS      = RAW / "Moffitt_and_Bambah-Mukku_et_al_merfish_all_cells.csv"
PATH_BOUNDARIES = RAW / "cellboundaries_example_animal.csv"
PATH_BARCODES   = RAW / "merfish_barcodes_example.csv"

for p in (PATH_CELLS, PATH_BOUNDARIES, PATH_BARCODES):
    print(f"{'found  ' if p.exists() else 'MISSING'}  {p}")


# A boundary coordinate is treated as truncated when the export cut it short:
# it ends in a bare decimal point ('-3216.'), has no decimal point at all
# ('-3208'), or keeps fewer than two decimal digits where three are standard.
def is_truncated(value_str):
    v = value_str.strip()
    if v.endswith("."):
        return True
    if "." not in v:
        return True
    decimal_part = v.split(".")[-1]
    if len(decimal_part) < 2:
        return True
    return False


## Step 1 — Clean the three tables

In [ ]:
# =============================================================
# STEP 1 — Clean the cell, boundary and barcode tables
# Cells: drop the 153,080 "Ambiguous" cells, the five Blank_* control
#   columns, and Fos (an activity marker imaged only in the behaviour
#   cohort, not a cell-identity gene).
# Boundaries: drop null polygons, rename feature_uID -> Cell_ID.
# Barcodes: drop blank-probe detections and molecules assigned to no
#   cell, rename Cell_name -> Cell_ID.
# =============================================================


# =====================================================================
#  1. CLEAN CELL DATA
# =====================================================================
print("=" * 60)
print("  1. CLEANING CELL DATA")
print("=" * 60)

df_cells = pd.read_csv(PATH_CELLS)
print(f"Original: {df_cells.shape[0]:,} rows × {df_cells.shape[1]} cols")

# --- Drop Ambiguous cells ---
n_ambiguous = (df_cells["Cell_class"] == "Ambiguous").sum()
df_cells = df_cells[df_cells["Cell_class"] != "Ambiguous"].copy()
print(f"Dropped {n_ambiguous:,} Ambiguous cells → {df_cells.shape[0]:,} rows")

# --- Drop Blank control columns (Blank_1 through Blank_5) ---
blank_cols = [c for c in df_cells.columns if c.startswith("Blank_")]
df_cells.drop(columns=blank_cols, inplace=True)
print(f"Dropped {len(blank_cols)} Blank columns: {blank_cols}")

# --- Drop Fos (cFos activity marker, not a cell identity gene) ---
if "Fos" in df_cells.columns:
    df_cells.drop(columns=["Fos"], inplace=True)
    print("Dropped Fos (cFos activity marker, 47% null, not cell identity)")

# --- Identify final column groups ---
metadata_cols = ["Cell_ID", "Animal_ID", "Animal_sex", "Behavior",
                 "Bregma", "Centroid_X", "Centroid_Y",
                 "Cell_class", "Neuron_cluster_ID"]
gene_cols = [c for c in df_cells.columns if c not in metadata_cols]

print(f"\nCleaned cell data: {df_cells.shape[0]:,} rows × {df_cells.shape[1]} cols")
print(f"  Metadata columns: {len(metadata_cols)}")
print(f"  Gene feature columns: {len(gene_cols)}")

# --- Verify no nulls in gene columns ---
gene_nulls = df_cells[gene_cols].isnull().sum().sum()
print(f"  Nulls in gene columns: {gene_nulls}")

# --- Class distribution after cleaning ---
print(f"\nCell_class distribution ({df_cells['Cell_class'].nunique()} classes):")
class_counts = df_cells["Cell_class"].value_counts()
for cls, cnt in class_counts.items():
    pct = 100 * cnt / len(df_cells)
    print(f"  {cls:<25} {cnt:>10,}  ({pct:>5.1f}%)")


# =====================================================================
#  2. CLEAN CELL BOUNDARIES
# =====================================================================
print("\n" + "=" * 60)
print("  2. CLEANING CELL BOUNDARIES")
print("=" * 60)

df_bounds = pd.read_csv(PATH_BOUNDARIES)
print(f"Original: {df_bounds.shape[0]:,} rows × {df_bounds.shape[1]} cols")

# --- Drop rows with null boundaries ---
n_null_bounds = df_bounds["boundaryX"].isnull().sum()
df_bounds = df_bounds.dropna(subset=["boundaryX", "boundaryY"]).copy()
print(f"Dropped {n_null_bounds} rows with null boundaries → {df_bounds.shape[0]:,} rows")

# --- Rename feature_uID to Cell_ID for consistency ---
df_bounds.rename(columns={"feature_uID": "Cell_ID"}, inplace=True)
print("Renamed feature_uID → Cell_ID")

# --- Check join overlap with cell data ---
common_ids = set(df_bounds["Cell_ID"]) & set(df_cells["Cell_ID"])
print(f"Boundary cells matching cell data: {len(common_ids):,} / {df_bounds.shape[0]:,}")

print(f"\nCleaned boundaries: {df_bounds.shape[0]:,} rows × {df_bounds.shape[1]} cols")


# =====================================================================
#  3. CLEAN MOLECULE BARCODES
# =====================================================================
print("\n" + "=" * 60)
print("  3. CLEANING MOLECULE BARCODES")
print("=" * 60)

df_barcodes = pd.read_csv(PATH_BARCODES)
print(f"Original: {df_barcodes.shape[0]:,} rows × {df_barcodes.shape[1]} cols")

# --- Drop Blank probe detections ---
blank_mask = df_barcodes["Gene_name"].str.startswith("Blank")
n_blank_molecules = blank_mask.sum()
df_barcodes = df_barcodes[~blank_mask].copy()
print(f"Dropped {n_blank_molecules:,} Blank probe detections → {df_barcodes.shape[0]:,} rows")

# --- Drop unassigned molecules (Cell_name is null) ---
n_unassigned = df_barcodes["Cell_name"].isnull().sum()
df_barcodes = df_barcodes.dropna(subset=["Cell_name"]).copy()
print(f"Dropped {n_unassigned:,} unassigned molecules → {df_barcodes.shape[0]:,} rows")

# --- Rename Cell_name to Cell_ID for consistency ---
df_barcodes.rename(columns={"Cell_name": "Cell_ID"}, inplace=True)
print("Renamed Cell_name → Cell_ID")

print(f"\nCleaned barcodes: {df_barcodes.shape[0]:,} rows × {df_barcodes.shape[1]} cols")
print(f"  Unique genes: {df_barcodes['Gene_name'].nunique()}")
print(f"  Unique cells: {df_barcodes['Cell_ID'].nunique()}")


# =====================================================================
#  4. SAVE CLEANED DATASETS
# =====================================================================
print("\n" + "=" * 60)
print("  4. SAVING CLEANED DATASETS")
print("=" * 60)

df_cells.to_csv(CLEAN / "cells_cleaned.csv", index=False)
print(f"Saved cells_cleaned.csv ({df_cells.shape[0]:,} × {df_cells.shape[1]})")

df_bounds.to_csv(CLEAN / "boundaries_cleaned.csv", index=False)
print(f"Saved boundaries_cleaned.csv ({df_bounds.shape[0]:,} × {df_bounds.shape[1]})")

df_barcodes.to_csv(CLEAN / "barcodes_cleaned.csv", index=False)
print(f"Saved barcodes_cleaned.csv ({df_barcodes.shape[0]:,} × {df_barcodes.shape[1]})")


# =====================================================================
#  5. FINAL SUMMARY
# =====================================================================
print("\n" + "=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"  Cells:      {df_cells.shape[0]:>10,} rows × {df_cells.shape[1]:>4} cols  |  {len(gene_cols)} gene features, 15 classes")
print(f"  Boundaries: {df_bounds.shape[0]:>10,} rows × {df_bounds.shape[1]:>4} cols  |  1 example animal")
print(f"  Barcodes:   {df_barcodes.shape[0]:>10,} rows × {df_barcodes.shape[1]:>4} cols  |  assigned molecules only")
print(f"\n  Removed:")
print(f"    - {n_ambiguous:,} Ambiguous cells (no confident class assignment)")
print(f"    - {len(blank_cols)} Blank control columns + Fos activity marker")
print(f"    - {n_null_bounds} boundary rows with null polygons")
print(f"    - {n_blank_molecules:,} Blank probe molecules + {n_unassigned:,} unassigned molecules")

## Step 2 — Link cells to boundaries

In [ ]:
# =============================================================
# STEP 2 — Link cells to boundaries
# Inner join of the cleaned cell table with the cleaned boundary table
# on Cell_ID. Boundaries exist for one animal only, so 64,312 of the
# 874,768 cells are linked.
# =============================================================

# ─── Load cleaned datasets ───
df_cells = pd.read_csv(CLEAN / "cells_cleaned.csv")
df_bounds = pd.read_csv(CLEAN / "boundaries_cleaned.csv")

print("=" * 60)
print("  LINKING CELLS ↔ BOUNDARIES")
print("=" * 60)
print(f"Cells:      {df_cells.shape[0]:,} rows × {df_cells.shape[1]} cols")
print(f"Boundaries: {df_bounds.shape[0]:,} rows × {df_bounds.shape[1]} cols")

# ─── Inner join on Cell_ID ───
df_linked = df_cells.merge(df_bounds, on="Cell_ID", how="inner")

print(f"\nLinked:     {df_linked.shape[0]:,} rows × {df_linked.shape[1]} cols")
print(f"Cells with boundaries:    {df_linked.shape[0]:,} / {df_cells.shape[0]:,} ({100*df_linked.shape[0]/df_cells.shape[0]:.1f}%)")
print(f"Boundaries matched:       {df_linked.shape[0]:,} / {df_bounds.shape[0]:,} ({100*df_linked.shape[0]/df_bounds.shape[0]:.1f}%)")

# ─── What animals are covered? ───
print(f"\n--- Animals in linked data ---")
print(f"Unique Animal_IDs: {df_linked['Animal_ID'].nunique()}")
print(f"Animal_IDs: {sorted(df_linked['Animal_ID'].unique())}")

# ─── Class distribution in linked subset ───
print(f"\n--- Cell_class distribution (linked subset) ---")
class_counts = df_linked["Cell_class"].value_counts()
for cls, cnt in class_counts.items():
    pct = 100 * cnt / len(df_linked)
    print(f"  {cls:<25} {cnt:>8,}  ({pct:>5.1f}%)")

# ─── Bregma slices covered ───
print(f"\n--- Bregma slices in linked data ---")
print(f"Unique Bregma values: {sorted(df_linked['Bregma'].unique())}")

# ─── Quick look at boundary format ───
print(f"\n--- Boundary data sample ---")
sample = df_linked[["Cell_ID", "boundaryX", "boundaryY"]].head(3)
for i, row in sample.iterrows():
    x_points = str(row["boundaryX"]).split(";")
    y_points = str(row["boundaryY"]).split(";")
    print(f"\n  Cell: {row['Cell_ID'][:20]}...")
    print(f"  boundaryX points: {len(x_points)}")
    print(f"  boundaryY points: {len(y_points)}")
    print(f"  X range: [{min(float(v) for v in x_points):.1f}, {max(float(v) for v in x_points):.1f}]")
    print(f"  Y range: [{min(float(v) for v in y_points):.1f}, {max(float(v) for v in y_points):.1f}]")

# ─── Save linked dataset ───
df_linked.to_csv(CLEAN / "cells_with_boundaries.csv", index=False)
print(f"\nSaved cells_with_boundaries.csv ({df_linked.shape[0]:,} × {df_linked.shape[1]})")

# ─── Summary ───
print(f"\n{'=' * 60}")
print(f"  SUMMARY")
print(f"{'=' * 60}")
print(f"  Full cell data (no boundaries):  {df_cells.shape[0]:,} cells, 15 classes")
print(f"  Linked subset (with boundaries): {df_linked.shape[0]:,} cells, {df_linked['Cell_class'].nunique()} classes")
print(f"  Cells WITHOUT boundaries:        {df_cells.shape[0] - df_linked.shape[0]:,} cells")

## Step 3 — Diagnose truncated coordinates

In [ ]:
# =============================================================
# STEP 3 — Diagnose truncated boundary coordinates (diagnostic only)
# Counts, for the last X and last Y value of every polygon, which of the
# three truncation forms it shows. Changes no data.
# =============================================================
df = pd.read_csv(CLEAN / "cells_with_boundaries.csv")

results = {
    "x_truncated": 0,
    "y_truncated": 0,
    "both_truncated": 0,
    "neither_truncated": 0,
    "total": len(df),
}

# Track truncation patterns
x_patterns = []
y_patterns = []


def truncation_reason(value_str):
    """Return (is_truncated, pattern) for one coordinate string."""
    v = value_str.strip()
    if v.endswith("."):
        return True, "ends_with_dot"
    if "." not in v:
        return True, "no_decimal"
    # Check decimal digit count (most values have 3 decimals)
    decimal_part = v.split(".")[-1]
    if len(decimal_part) < 2:
        return True, f"short_decimal({len(decimal_part)})"
    return False, "ok"


for _, row in df.iterrows():
    x_vals = str(row["boundaryX"]).split(";")
    y_vals = str(row["boundaryY"]).split(";")

    last_x = x_vals[-1]
    last_y = y_vals[-1]

    x_trunc, x_reason = truncation_reason(last_x)
    y_trunc, y_reason = truncation_reason(last_y)

    x_patterns.append(x_reason)
    y_patterns.append(y_reason)

    if x_trunc and y_trunc:
        results["both_truncated"] += 1
    elif x_trunc:
        results["x_truncated"] += 1
    elif y_trunc:
        results["y_truncated"] += 1
    else:
        results["neither_truncated"] += 1

print("=" * 50)
print("  TRUNCATION DETECTION RESULTS")
print("=" * 50)
print(f"Total cells:            {results['total']:,}")
print(f"X only truncated:       {results['x_truncated']:,}")
print(f"Y only truncated:       {results['y_truncated']:,}")
print(f"Both truncated:         {results['both_truncated']:,}")
print(f"Neither truncated:      {results['neither_truncated']:,}")

print(f"\n--- Last X value patterns ---")
x_s = pd.Series(x_patterns)
for pattern, cnt in x_s.value_counts().items():
    print(f"  {pattern:<25} {cnt:>8,}  ({100*cnt/len(df):.1f}%)")

print(f"\n--- Last Y value patterns ---")
y_s = pd.Series(y_patterns)
for pattern, cnt in y_s.value_counts().items():
    print(f"  {pattern:<25} {cnt:>8,}  ({100*cnt/len(df):.1f}%)")

# Show some examples of each category
print(f"\n--- Examples: neither truncated ---")
for _, row in df.iterrows():
    x_vals = str(row["boundaryX"]).split(";")
    y_vals = str(row["boundaryY"]).split(";")
    x_ok = not truncation_reason(x_vals[-1])[0]
    y_ok = not truncation_reason(y_vals[-1])[0]
    if x_ok and y_ok:
        print(f"  X({len(x_vals)} pts) last: '{x_vals[-1]}'  |  Y({len(y_vals)} pts) last: '{y_vals[-1]}'")
        break

print(f"\n--- Examples: X truncated only ---")
for _, row in df.iterrows():
    x_vals = str(row["boundaryX"]).split(";")
    y_vals = str(row["boundaryY"]).split(";")
    x_trunc = truncation_reason(x_vals[-1])[0]
    y_ok = not truncation_reason(y_vals[-1])[0]
    if x_trunc and y_ok:
        print(f"  X({len(x_vals)} pts) last: '{x_vals[-1]}'  |  Y({len(y_vals)} pts) last: '{y_vals[-1]}'")
        break

print(f"\n--- Examples: Y truncated only ---")
for _, row in df.iterrows():
    x_vals = str(row["boundaryX"]).split(";")
    y_vals = str(row["boundaryY"]).split(";")
    x_ok = not truncation_reason(x_vals[-1])[0]
    y_trunc = truncation_reason(y_vals[-1])[0]
    if x_ok and y_trunc:
        print(f"  X({len(x_vals)} pts) last: '{x_vals[-1]}'  |  Y({len(y_vals)} pts) last: '{y_vals[-1]}'")
        break

print(f"\n--- Examples: both truncated ---")
for _, row in df.iterrows():
    x_vals = str(row["boundaryX"]).split(";")
    y_vals = str(row["boundaryY"]).split(";")
    x_trunc = truncation_reason(x_vals[-1])[0]
    y_trunc = truncation_reason(y_vals[-1])[0]
    if x_trunc and y_trunc:
        print(f"  X({len(x_vals)} pts) last: '{x_vals[-1]}'  |  Y({len(y_vals)} pts) last: '{y_vals[-1]}'")
        break

## Step 4 — Count differences after the repair

In [ ]:
# =============================================================
# STEP 4 — X/Y count differences after the repair (diagnostic only)
# After dropping truncated last values, how far apart are the X and Y
# counts? Differences of 0 and 1 form one peak, separated by a trough at
# 2 from a long tail — the basis for the tolerance of 1 used in Step 5.
# =============================================================
df = pd.read_csv(CLEAN / "cells_with_boundaries.csv")

diffs = []

for _, row in df.iterrows():
    x_vals = str(row["boundaryX"]).split(";")
    y_vals = str(row["boundaryY"]).split(";")

    if is_truncated(x_vals[-1]):
        x_vals = x_vals[:-1]
    if is_truncated(y_vals[-1]):
        y_vals = y_vals[:-1]

    diffs.append(abs(len(x_vals) - len(y_vals)))

s = pd.Series(diffs)
total = len(s)

print("=" * 50)
print("  DIFFERENCE DISTRIBUTION (after truncation fix)")
print("=" * 50)
print(f"Total cells: {total:,}\n")

print(f"  diff == 0:   {(s == 0).sum():>8,}  ({100*(s == 0).sum()/total:.1f}%)")
print(f"  diff <= 1:   {(s <= 1).sum():>8,}  ({100*(s <= 1).sum()/total:.1f}%)")
print(f"  diff <= 2:   {(s <= 2).sum():>8,}  ({100*(s <= 2).sum()/total:.1f}%)")
print(f"  diff <= 3:   {(s <= 3).sum():>8,}  ({100*(s <= 3).sum()/total:.1f}%)")
print(f"  diff <= 5:   {(s <= 5).sum():>8,}  ({100*(s <= 5).sum()/total:.1f}%)")
print(f"  diff <= 10:  {(s <= 10).sum():>8,}  ({100*(s <= 10).sum()/total:.1f}%)")
print(f"  diff > 10:   {(s > 10).sum():>8,}  ({100*(s > 10).sum()/total:.1f}%)")

print(f"\n--- Full distribution (first 20) ---")
vc = s.value_counts().sort_index()
for diff_val, cnt in vc.head(20).items():
    print(f"  diff = {diff_val:<4}  {cnt:>6,}  ({100*cnt/total:.1f}%)")

## Step 5 — Repair boundaries and save

In [ ]:
# =============================================================
# STEP 5 — Repair boundaries and save the boundary-annotated subset
#   1. Drop the last X / Y value only if it is truncated
#   2. Keep only cells whose X and Y counts differ by at most 1
#   3. Pair the two strings to the shorter length
#   4. Force-close each polygon (append the first point to the end)
#   5. Save -> data/clean/cells_boundaries_clean.csv (35,522 cells)
# =============================================================
df = pd.read_csv(CLEAN / "cells_with_boundaries.csv")
print(f"Input: {len(df):,} cells")

clean_rows = []

for _, row in df.iterrows():
    x_vals = str(row["boundaryX"]).split(";")
    y_vals = str(row["boundaryY"]).split(";")

    # Drop truncated last values
    if is_truncated(x_vals[-1]):
        x_vals = x_vals[:-1]
    if is_truncated(y_vals[-1]):
        y_vals = y_vals[:-1]

    diff = abs(len(x_vals) - len(y_vals))

    # Keep only diff <= 1
    if diff > 1:
        continue

    # Pair to min length
    n = min(len(x_vals), len(y_vals))
    x_vals = x_vals[:n]
    y_vals = y_vals[:n]

    # Force-close: append first point to end if not already closed
    if x_vals[0] != x_vals[-1] or y_vals[0] != y_vals[-1]:
        x_vals.append(x_vals[0])
        y_vals.append(y_vals[0])

    # Store clean boundary strings
    row_copy = row.copy()
    row_copy["boundaryX"] = ";".join(x_vals)
    row_copy["boundaryY"] = ";".join(y_vals)
    clean_rows.append(row_copy)

df_clean = pd.DataFrame(clean_rows)

# ─── Verify all X/Y counts are equal ───
mismatch = 0
for _, row in df_clean.iterrows():
    nx = len(str(row["boundaryX"]).split(";"))
    ny = len(str(row["boundaryY"]).split(";"))
    if nx != ny:
        mismatch += 1

# ─── Class distribution ───
print(f"\n{'=' * 60}")
print(f"  CLEAN LINKED DATASET")
print(f"{'=' * 60}")
print(f"Cells:       {len(df_clean):,} / {len(df):,} ({100*len(df_clean)/len(df):.1f}%)")
print(f"Columns:     {df_clean.shape[1]}")
print(f"X/Y mismatches remaining: {mismatch}")

print(f"\nCell_class distribution ({df_clean['Cell_class'].nunique()} classes):")
for cls, cnt in df_clean["Cell_class"].value_counts().items():
    pct = 100 * cnt / len(df_clean)
    print(f"  {cls:<25} {cnt:>8,}  ({pct:>5.1f}%)")

# ─── Polygon stats ───
pts = df_clean["boundaryX"].apply(lambda s: len(str(s).split(";")))
print(f"\nPolygon point counts (including closure point):")
print(f"  Min: {pts.min()}  Max: {pts.max()}  Mean: {pts.mean():.1f}  Median: {pts.median():.0f}")

# ─── Save ───
df_clean.to_csv(CLEAN / "cells_boundaries_clean.csv", index=False)
print(f"\nSaved cells_boundaries_clean.csv ({len(df_clean):,} × {df_clean.shape[1]})")